In [ ]:
%pip install -q numpy matplotlib sagemaker boto3 pandas scikit-learn Pillow

# Week 2 Monday -- Deploy, Evaluate, and Sequence Models

On Friday you built two models on SageMaker: a **Random Forest** for tabular fraud detection (Script Mode) and a **CNN image classifier** for CIFAR-10 (JumpStart). Both produced `model.tar.gz` artifacts in S3, but neither was ever deployed or evaluated on held-out data.

Today you close that loop. By the end of this notebook you will have:

1. **Deployed** the Random Forest to a real-time endpoint and **evaluated** it with precision, recall, F1, and a confusion matrix on the validation set.
2. **Opened the LSTM black box** from Friday's encoder-decoder -- understanding gates, cell state, and the vanishing gradient problem.
3. **Deployed and evaluated** the CNN image classifier on CIFAR-10 test data.
4. Learned the **MLOps concepts** (pipelines, DAGs, CI/CD for ML) that automate the manual workflow you just performed.

| Block | Content | Minutes |
|-------|---------|--------|
| Stage 1 | Model Registry, Deployment, and Evaluation | 60 |
| Break 1 | Stretch / Questions | 5 |
| Stage 2 | RNNs, LSTMs, and GRUs | 55 |
| Break 2 | Stretch / Questions | 5 |
| Stage 3 | CNN Evaluation + MLOps Concepts | 45 |


## Setup

Run the cell below to establish your SageMaker session. This connects to your execution role, default S3 bucket, and region.

In [ ]:
import boto3
import sagemaker

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
bucket = session.default_bucket()
prefix = "fraudshield"

sm_client = boto3.client("sagemaker")
s3 = boto3.client("s3")

print(f"Region:  {region}")
print(f"Bucket:  s3://{bucket}")
print(f"Role:    ...{role[-30:]}")

---
# STAGE 1 -- Model Registry, Deployment, and Evaluation

**Connecting to Friday:** On Friday you trained a Random Forest classifier on synthetic fraud data using SageMaker Script Mode. The training job produced a `model.tar.gz` artifact in S3. That artifact contains a serialized `model.pkl` (scikit-learn `RandomForestClassifier`). You also generated `validation.csv` -- 400 rows of fraud data the model has never seen.

Now we deploy that model and find out how well it actually works.

## STEP 1 -- Verify Friday's Artifacts

Before we can deploy, we need to confirm the model artifact and validation data exist in S3. The cell below checks for the Random Forest artifact. If it is missing, a fallback cell regenerates the data and retrains.

> **What is happening:** We list S3 objects under the `fraudshield/output/` prefix to find the `model.tar.gz` from Friday's training job. We also check for the validation CSV.

In [ ]:
import json

s3 = boto3.client("s3")
sm_client = boto3.client("sagemaker")

rf_model_uri = None
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=bucket, Prefix=f"{prefix}/output/"):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith("model.tar.gz") and "cnn" not in obj["Key"].lower():
            rf_model_uri = f"s3://{bucket}/{obj['Key']}"

val_exists = False
try:
    s3.head_object(Bucket=bucket, Key=f"{prefix}/data/validation/validation.csv")
    val_exists = True
except Exception:
    pass

print(f"RF model artifact: {rf_model_uri or 'NOT FOUND'}")
print(f"Validation CSV:    {'Found' if val_exists else 'NOT FOUND'}")

if rf_model_uri and val_exists:
    print("\nAll artifacts present. Proceed to Step 2.")
else:
    print("\nMissing artifacts -- run the fallback cell below.")

### Retrain the Model Inside the SageMaker Container

The model from Friday's Studio notebook was trained interactively in JupyterLab, which uses whatever scikit-learn version is installed in the kernel (1.3+). But the SageMaker inference container runs sklearn **1.2.x**. The pickle format changed between these versions (the decision tree node array gained a `missing_go_to_left` field in 1.3), so the Studio-trained model cannot be loaded by the inference container.

The cell below retrains the model using a **managed training job** with `framework_version="1.2-1"`. This runs the training inside the same container version used for inference, guaranteeing the model pickle is compatible. This takes approximately 5 minutes.

> **Lesson:** Always match your training and inference environments. In production, this is handled by CI/CD pipelines that use the same container image for both.

In [ ]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
n = 2000
data = pd.DataFrame({
    "amount": np.random.exponential(500, n).round(2),
    "hour": np.random.randint(0, 24, n),
    "distance_from_home": np.random.exponential(50, n).round(2),
    "transaction_count_24h": np.random.poisson(5, n),
    "is_international": np.random.binomial(1, 0.1, n),
    "merchant_risk_score": np.random.uniform(0, 1, n).round(3),
})
data["target"] = ((data["amount"] > 800) & (data["hour"] < 6) | (data["merchant_risk_score"] > 0.85)).astype(int)
noise = np.random.random(n) < 0.08
data["target"] = (data["target"] ^ noise.astype(int))
train = data.iloc[:1600]
val = data.iloc[1600:]
train.to_csv("train.csv", index=False)
val.to_csv("validation.csv", index=False)

s3.upload_file("train.csv", bucket, f"{prefix}/data/train/train.csv")
s3.upload_file("validation.csv", bucket, f"{prefix}/data/validation/validation.csv")

full = pd.concat([train, val], axis=0, ignore_index=True)
full.to_csv("fraud_full.csv", index=False)
s3.upload_file("fraud_full.csv", bucket, f"{prefix}/data/fraud_full.csv")
print(f"Also uploaded combined dataset: s3://{bucket}/{prefix}/data/fraud_full.csv (for Week 2 Wednesday experiments)")

os.makedirs("code", exist_ok=True)
with open("code/train.py", "w", newline="\n") as f:
    f.write('''import argparse, os, pandas as pd, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--random-state", type=int, default=42)
    args = parser.parse_args()
    train_dir = os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train")
    model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
    data = pd.read_csv(os.path.join(train_dir, "train.csv"))
    X = data.drop("target", axis=1)
    y = data["target"]
    model = RandomForestClassifier(n_estimators=args.n_estimators, random_state=args.random_state)
    model.fit(X, y)
    print(f"Training accuracy: {accuracy_score(y, model.predict(X)):.4f}")
    joblib.dump(model, os.path.join(model_dir, "model.pkl"))

if __name__ == "__main__":
    main()
''')

from sagemaker.sklearn import SKLearn
sklearn_estimator = SKLearn(
    entry_point="train.py",
    source_dir="code/",
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    framework_version="1.2-1",
    sagemaker_session=session,
    output_path=f"s3://{bucket}/{prefix}/output/",
    hyperparameters={"n-estimators": 100, "random-state": 42},
)
sklearn_estimator.fit({"train": f"s3://{bucket}/{prefix}/data/train/"}, wait=True, logs="All")
rf_model_uri = sklearn_estimator.model_data
print(f"\nModel artifact: {rf_model_uri}")

## STEP 1b -- Prepare the Model Artifact for Serving

Since we retrained with the managed training job above, the model pickle is now compatible with the `1.2-1` inference container. But the sklearn inference container still needs an **inference script** -- a Python module that tells it how to load and serve the model.

The container looks for a module name in the `SAGEMAKER_PROGRAM` environment variable. When you deploy directly from an `SKLearn` estimator, the SDK sets this automatically. But since we are deploying through the **Model Registry** (the production pattern), we must provide it ourselves.

The cell below downloads the model artifact, injects `code/inference.py` with four functions (`model_fn`, `input_fn`, `predict_fn`, `output_fn`), repackages, and uploads.

> **What is happening:** We are adding a serving script to the model archive. Without `SAGEMAKER_PROGRAM` pointing to a valid module, the container crashes on every health check with `AttributeError: 'NoneType' object has no attribute 'startswith'`.

In [ ]:
import tarfile
import tempfile
import shutil
import os

work_dir = tempfile.mkdtemp()
extract_dir = os.path.join(work_dir, "extract")
os.makedirs(extract_dir)
local_tar = os.path.join(work_dir, "model.tar.gz")

rf_s3_key = rf_model_uri.replace(f"s3://{bucket}/", "")
s3.download_file(bucket, rf_s3_key, local_tar)

with tarfile.open(local_tar, "r:gz") as tar:
    tar.extractall(extract_dir)

print("Original model.tar.gz contents:")
for f in sorted(os.listdir(extract_dir)):
    print(f"  {f}")

code_dir = os.path.join(extract_dir, "code")
os.makedirs(code_dir, exist_ok=True)

with open(os.path.join(code_dir, "inference.py"), "w", newline="\n") as f:
    f.write(
"""import os
import joblib
import numpy as np


def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "model.pkl"))


def input_fn(request_body, request_content_type):
    if request_content_type == "text/csv":
        values = [float(x) for x in request_body.strip().split(",")]
        return np.array(values).reshape(1, -1)
    raise ValueError(f"Unsupported content type: {request_content_type}")


def predict_fn(input_data, model):
    return model.predict(input_data)


def output_fn(prediction, accept):
    return str(int(prediction[0]))
""")

repackaged_tar = os.path.join(work_dir, "model-deploy.tar.gz")
with tarfile.open(repackaged_tar, "w:gz") as tar:
    for item in os.listdir(extract_dir):
        tar.add(os.path.join(extract_dir, item), arcname=item)

deploy_key = rf_s3_key.rsplit("/", 1)[0] + "/model-deploy.tar.gz"
s3.upload_file(repackaged_tar, bucket, deploy_key)
rf_model_uri = f"s3://{bucket}/{deploy_key}"

shutil.rmtree(work_dir)

print(f"\nRepackaged model artifact: {rf_model_uri}")
print("Contents: model.pkl + code/inference.py")

## STEP 2 -- Register the Model in Model Registry

**What is Model Registry?** Think of it as "Git for models." A **Model Package Group** is like a repository -- it holds all versions of a particular model. Each **Model Package** (version) captures:

- The S3 path to the model artifact (`model.tar.gz`)
- The Docker container image for inference
- Supported instance types
- Approval status (the deployment gate)

**Why does this matter?** Without a registry, you end up with dozens of `model.tar.gz` files in S3 with no way to know which one is in production, which was approved, or what accuracy it achieved. The registry solves this.

> **What is happening in the cell below:** We create a Model Package Group called `fraudshield-rf`, then register Friday's trained model as version 1 with `PendingManualApproval` status.

In [ ]:
from sagemaker import image_uris

MODEL_PACKAGE_GROUP = "fraudshield-rf"

sklearn_image = image_uris.retrieve("sklearn", region, version="1.2-1")

try:
    sm_client.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelPackageGroupDescription="FraudShield Random Forest fraud detection models",
    )
    print(f"Created Model Package Group: {MODEL_PACKAGE_GROUP}")
except sm_client.exceptions.ClientError as e:
    if "already exists" in str(e):
        print(f"Model Package Group already exists: {MODEL_PACKAGE_GROUP}")
    else:
        raise

register_response = sm_client.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    ModelPackageDescription="RF v1 -- trained on synthetic fraud data, 100 estimators",
    InferenceSpecification={
        "Containers": [{
            "Image": sklearn_image,
            "ModelDataUrl": rf_model_uri,
            "Environment": {
                "SAGEMAKER_PROGRAM": "inference.py",
                "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
            },
        }],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.xlarge"],
    },
    ModelApprovalStatus="PendingManualApproval",
)

model_package_arn = register_response["ModelPackageArn"]
print(f"\nRegistered Model Package: {model_package_arn}")
print(f"  Status:    PendingManualApproval")
print(f"  Artifact:  {rf_model_uri}")
print(f"  Container: {sklearn_image}")

## STEP 3 -- Approve and Deploy the Model

In a production setting, approval would follow a review process: a senior data scientist checks the model's performance metrics, validates the training data, and signs off. Here we approve immediately to focus on the deployment mechanics.

**The three-object deployment pattern:**

| Object | Purpose |
|--------|---------|
| **Model** | Links the artifact in S3 to the inference container |
| **Endpoint Configuration** | Specifies instance type, count, and optional data capture |
| **Endpoint** | The live service that accepts inference requests |

The SageMaker SDK's `.deploy()` method creates all three for you.

> **What is happening:** We approve the model version, then deploy it to a real-time endpoint. Deployment takes 3-5 minutes while SageMaker provisions an instance and loads the model.

In [ ]:
sm_client.update_model_package(
    ModelPackageArn=model_package_arn,
    ModelApprovalStatus="Approved",
    ApprovalDescription="Approved for Monday lecture evaluation",
)
print(f"Model approved: {model_package_arn}")

In [ ]:
from sagemaker.model import ModelPackage
from datetime import datetime

TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
RF_ENDPOINT_NAME = f"fraudshield-rf-{TIMESTAMP}"

rf_model = ModelPackage(
    role=role,
    model_package_arn=model_package_arn,
    sagemaker_session=session,
)

print(f"Deploying to endpoint: {RF_ENDPOINT_NAME}")
print("This takes 3-5 minutes. While we wait, review the deployment pattern above.")

rf_predictor = rf_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=RF_ENDPOINT_NAME,
)

print(f"\nEndpoint in service: {RF_ENDPOINT_NAME}")

## STEP 4 -- Invoke the Endpoint and Evaluate

This is the payoff. We load the validation data (400 rows the model has never seen), send each row to the live endpoint, collect predictions, and compute performance metrics.

**Why these metrics matter for FraudShield:**

| Metric | What it measures | FraudShield impact |
|--------|-----------------|--------------------|
| **Accuracy** | Overall correct predictions | Misleading if classes are imbalanced |
| **Precision** | Of predicted frauds, how many are real? | Low precision = too many false alarms, wasting investigator time |
| **Recall** | Of actual frauds, how many did we catch? | Low recall = missed frauds, real financial loss |
| **F1** | Harmonic mean of precision and recall | Balances both concerns |

> **What is happening:** We download `validation.csv`, send the feature columns (without the `target`) to the endpoint row by row, and compare the predictions to the actual labels.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

s3.download_file(bucket, f"{prefix}/data/validation/validation.csv", "validation.csv")
val_df = pd.read_csv("validation.csv")
X_val = val_df.drop("target", axis=1)
y_val = val_df["target"].values

print(f"Validation set: {len(val_df)} rows, {val_df['target'].mean():.1%} fraud rate")
print(f"Features: {list(X_val.columns)}")

In [ ]:
sm_runtime = boto3.client("sagemaker-runtime")

predictions = []
for _, row in X_val.iterrows():
    payload = ",".join(str(v) for v in row.values)
    response = sm_runtime.invoke_endpoint(
        EndpointName=RF_ENDPOINT_NAME,
        ContentType="text/csv",
        Body=payload,
    )
    pred = float(response["Body"].read().decode("utf-8").strip())
    predictions.append(int(pred))

predictions = np.array(predictions)
print(f"Predictions collected: {len(predictions)}")
print(f"Predicted fraud rate:  {predictions.mean():.1%}")
print(f"Actual fraud rate:     {y_val.mean():.1%}")

### Performance Metrics

Now we compute the metrics that tell us how well the model performs on unseen data. Pay attention to the difference between precision and recall -- for fraud detection, the trade-off between these two has direct business consequences.

> **Discussion:** If precision is high but recall is low, what does that mean for FraudShield's customers? What about the reverse?

In [ ]:
acc = accuracy_score(y_val, predictions)
prec = precision_score(y_val, predictions)
rec = recall_score(y_val, predictions)
f1 = f1_score(y_val, predictions)

print("=" * 50)
print("FraudShield RF Model -- Validation Performance")
print("=" * 50)
print(f"  Accuracy:  {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print()
print("Classification Report:")
print(classification_report(y_val, predictions, target_names=["Legitimate", "Fraud"]))

In [ ]:
cm = confusion_matrix(y_val, predictions)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=["Legitimate", "Fraud"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("FraudShield RF -- Confusion Matrix (Validation Set)")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (correctly identified legitimate):  {tn}")
print(f"False Positives (false alarms):                    {fp}")
print(f"False Negatives (missed fraud):                    {fn}")
print(f"True Positives (correctly caught fraud):           {tp}")

## STEP 5 -- Cleanup RF Endpoint

We are done evaluating the Random Forest. Delete the endpoint now to stop billing. We will deploy the CNN model in Stage 3.

> **Important:** Always delete in this order: endpoint first (stops billing), then endpoint configuration, then model.

In [ ]:
try:
    rf_predictor.delete_endpoint(delete_endpoint_config=True)
    print(f"Deleted endpoint: {RF_ENDPOINT_NAME}")
except Exception as e:
    print(f"Cleanup note: {e}")
    sm_client.delete_endpoint(EndpointName=RF_ENDPOINT_NAME)
    print(f"Deleted endpoint via client: {RF_ENDPOINT_NAME}")

---
# STAGE 2 -- RNNs, LSTMs, and GRUs

**Connecting to Friday:** On Friday you saw the encoder-decoder pattern for sequence-to-sequence tasks. The encoder and decoder both used LSTMs, but we treated them as black boxes with a brief primer. Now we open that box.

**What you will learn:**
- How recurrent neural networks process data step by step, sharing weights across time.
- Why vanilla RNNs fail on long sequences (vanishing gradients).
- How LSTM gates solve this with a cell state highway.
- How GRUs simplify the LSTM with fewer parameters.

After this stage, revisit Friday's encoder-decoder architecture -- the internals will make much more sense.

## STEP 7 -- From Images to Sequences

CNNs exploit **spatial** structure: a pixel's neighbors matter. But many data types have **temporal** structure:

- **Transaction sequences:** the order of purchases reveals spending patterns
- **Text:** word order determines meaning ("dog bites man" vs "man bites dog")
- **Time series:** stock prices, sensor readings, server logs

We need architectures that process data **step by step**, maintaining a **memory** of what came before. That is what recurrent neural networks do.

## STEP 8 -- Generate Synthetic Sentiment Data

We create a toy dataset to study RNN architectures without the complexity of real NLP. Each "sequence" is a list of integer tokens. Positive sequences tend to have higher-valued tokens; negative sequences tend to have lower-valued tokens. Noise tokens are injected to make the task non-trivial.

> **What is happening:** The generator creates 2,000 training and 400 test sequences, each of length 20, with a vocabulary of 50 tokens.

In [ ]:
import numpy as np

VOCAB_SIZE = 50
SEQ_LEN = 20
EMBED_DIM = 16
HIDDEN_DIM = 32
NUM_TRAIN = 2000
NUM_TEST = 400

def generate_sentiment_data(n_samples, seq_len, vocab_size, seed=42):
    rng = np.random.RandomState(seed)
    sequences, labels = [], []
    for _ in range(n_samples):
        label = rng.randint(0, 2)
        if label == 1:
            seq = rng.randint(vocab_size // 2, vocab_size, size=seq_len)
        else:
            seq = rng.randint(0, vocab_size // 2, size=seq_len)
        noise_idx = rng.choice(seq_len, size=seq_len // 4, replace=False)
        seq[noise_idx] = rng.randint(0, vocab_size, size=len(noise_idx))
        sequences.append(seq)
        labels.append(label)
    return np.array(sequences), np.array(labels)

X_train_seq, y_train_seq = generate_sentiment_data(NUM_TRAIN, SEQ_LEN, VOCAB_SIZE, seed=42)
X_test_seq, y_test_seq = generate_sentiment_data(NUM_TEST, SEQ_LEN, VOCAB_SIZE, seed=99)

print(f"Training sequences: {X_train_seq.shape}")
print(f"Test sequences:     {X_test_seq.shape}")
print(f"Vocabulary size:    {VOCAB_SIZE}")
print(f"Sequence length:    {SEQ_LEN}")
print(f"Class balance:      {y_train_seq.mean():.2f} positive")
print(f"\nSample sequence:    {X_train_seq[0]}")
print(f"Sample label:       {y_train_seq[0]} ({'positive' if y_train_seq[0] else 'negative'})")

## STEP 9 -- Vanilla RNN Forward Pass

A vanilla RNN processes a sequence one token at a time. At each step $t$:

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$

The same weight matrices $W_{xh}$ and $W_{hh}$ are used at every time step -- this is **weight sharing**. The hidden state $h_t$ carries information forward through the sequence.

After processing the entire sequence, we use the final hidden state $h_T$ to predict:

$$\hat{y} = \sigma(W_{hy} \cdot h_T + b_y)$$

> **What is happening:** We implement a single-layer RNN from scratch in NumPy. This is pedagogical -- in production you would use PyTorch or TensorFlow.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

class VanillaRNN:
    def __init__(self, vocab_size, embed_dim, hidden_dim, seed=42):
        rng = np.random.RandomState(seed)
        scale = 0.01
        self.embedding = rng.randn(vocab_size, embed_dim) * scale
        self.W_xh = rng.randn(embed_dim, hidden_dim) * scale
        self.W_hh = rng.randn(hidden_dim, hidden_dim) * scale
        self.b_h = np.zeros(hidden_dim)
        self.W_hy = rng.randn(hidden_dim, 1) * scale
        self.b_y = np.zeros(1)
        self.hidden_dim = hidden_dim

    def forward(self, sequence):
        """Process one sequence, return probability of positive class."""
        h = np.zeros(self.hidden_dim)
        for token in sequence:
            x = self.embedding[token]
            h = np.tanh(x @ self.W_xh + h @ self.W_hh + self.b_h)
        logit = h @ self.W_hy + self.b_y
        return sigmoid(logit[0]), h

rnn = VanillaRNN(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
prob, final_h = rnn.forward(X_train_seq[0])
print(f"Sample prediction probability: {prob:.4f}")
print(f"Final hidden state shape:      {final_h.shape}")
print(f"Final hidden state (first 8):  {final_h[:8].round(4)}")
print(f"\nNote: with random weights this prediction is meaningless.")
print(f"The point is to see the forward pass mechanics.")

## STEP 10 -- The Vanishing Gradient Problem

During backpropagation through time (BPTT), gradients flow backward through every time step. At each step, the gradient is multiplied by the weight matrix $W_{hh}$ and the derivative of $\tanh$.

If the largest singular value of $W_{hh}$ is less than 1, gradients **decay exponentially**. After 50 steps, the gradient from step 0 is negligible -- the network cannot learn from early tokens.

> **What is happening:** We simulate gradient flow through 50 time steps with a singular value of 0.9. The bar chart shows how the gradient at early time steps is a tiny fraction of the gradient at late steps.

In [ ]:
import matplotlib.pyplot as plt

seq_length = 50
W_singular_value = 0.9

gradient_norms = []
current_magnitude = 1.0
for t in range(seq_length):
    gradient_norms.append(current_magnitude)
    current_magnitude *= W_singular_value

gradient_norms = gradient_norms[::-1]

plt.figure(figsize=(10, 4))
plt.bar(range(seq_length), gradient_norms, color="steelblue")
plt.xlabel("Time Step (earlier --> later)")
plt.ylabel("Relative Gradient Magnitude")
plt.title(f"Gradient Flow in Vanilla RNN ({seq_length} steps, max singular value = {W_singular_value})")
plt.tight_layout()
plt.show()

print(f"Gradient at step 0 (earliest):  {gradient_norms[0]:.6f}")
print(f"Gradient at step 49 (latest):   {gradient_norms[49]:.6f}")
print(f"Ratio (earliest/latest):        {gradient_norms[0]/gradient_norms[49]:.6f}")
print(f"\nThe gradient at step 0 is {gradient_norms[0]/gradient_norms[49]*100:.2f}% of the gradient at step 49.")
print(f"The RNN effectively cannot learn from the first tokens in the sequence.")

> **Discussion:** What happens if the singular value is 1.1 instead of 0.9? (Exploding gradients -- the magnitudes grow exponentially. Gradient clipping helps but does not solve the fundamental representation problem.)

## STEP 11 -- LSTM: Gated Memory

The LSTM (Long Short-Term Memory) solves the vanishing gradient problem with a **cell state highway**. Information flows through the cell state via **addition**, not repeated multiplication. Four gates control the flow:

| Gate | Formula | Purpose |
|------|---------|--------|
| **Forget** | $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$ | What to erase from cell state |
| **Input** | $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$ | What new info to write |
| **Candidate** | $\tilde{c}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$ | The new info itself |
| **Output** | $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$ | What to expose as hidden state |

**Cell state update:** $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$

**Hidden state:** $h_t = o_t \odot \tanh(c_t)$

The key insight: the cell state update uses **element-wise addition**. Gradients flow through the addition unchanged, avoiding the vanishing gradient problem.

> **What is happening:** We implement a single LSTM cell in NumPy so you can see each gate in action.

In [ ]:
class LSTMCell:
    def __init__(self, input_dim, hidden_dim, seed=42):
        rng = np.random.RandomState(seed)
        scale = 0.01
        combined = input_dim + hidden_dim
        self.W_f = rng.randn(combined, hidden_dim) * scale
        self.b_f = np.zeros(hidden_dim)
        self.W_i = rng.randn(combined, hidden_dim) * scale
        self.b_i = np.zeros(hidden_dim)
        self.W_c = rng.randn(combined, hidden_dim) * scale
        self.b_c = np.zeros(hidden_dim)
        self.W_o = rng.randn(combined, hidden_dim) * scale
        self.b_o = np.zeros(hidden_dim)
        self.hidden_dim = hidden_dim

    def step(self, x, h_prev, c_prev):
        """One LSTM time step. Returns (h_next, c_next) and gate values for inspection."""
        combined = np.concatenate([h_prev, x])
        f = sigmoid(combined @ self.W_f + self.b_f)
        i = sigmoid(combined @ self.W_i + self.b_i)
        c_candidate = np.tanh(combined @ self.W_c + self.b_c)
        o = sigmoid(combined @ self.W_o + self.b_o)
        c_next = f * c_prev + i * c_candidate
        h_next = o * np.tanh(c_next)
        return h_next, c_next, {"forget": f, "input": i, "candidate": c_candidate, "output": o}

class LSTMClassifier:
    def __init__(self, vocab_size, embed_dim, hidden_dim, seed=42):
        rng = np.random.RandomState(seed)
        self.embedding = rng.randn(vocab_size, embed_dim) * 0.01
        self.lstm = LSTMCell(embed_dim, hidden_dim, seed=seed)
        self.W_out = rng.randn(hidden_dim, 1) * 0.01
        self.b_out = np.zeros(1)
        self.hidden_dim = hidden_dim

    def forward(self, sequence):
        h = np.zeros(self.hidden_dim)
        c = np.zeros(self.hidden_dim)
        gate_history = []
        for token in sequence:
            x = self.embedding[token]
            h, c, gates = self.lstm.step(x, h, c)
            gate_history.append(gates)
        logit = h @ self.W_out + self.b_out
        return sigmoid(logit[0]), h, c, gate_history

lstm_model = LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
prob, h_final, c_final, gates = lstm_model.forward(X_train_seq[0])

print(f"LSTM prediction probability: {prob:.4f}")
print(f"Final hidden state norm:     {np.linalg.norm(h_final):.4f}")
print(f"Final cell state norm:       {np.linalg.norm(c_final):.4f}")
print(f"\nGate values at last time step:")
for name, vals in gates[-1].items():
    print(f"  {name:10s} mean={vals.mean():.4f}  std={vals.std():.4f}")

## STEP 12 -- GRU: Simplified Gating

The GRU (Gated Recurrent Unit) simplifies the LSTM by merging the forget and input gates into a single **update gate** $z_t$, and using a **reset gate** $r_t$ to control how much of the previous hidden state to expose:

| Gate | Formula | Purpose |
|------|---------|--------|
| **Update** | $z_t = \sigma(W_z \cdot [h_{t-1}, x_t])$ | Balance old vs new (like forget + input combined) |
| **Reset** | $r_t = \sigma(W_r \cdot [h_{t-1}, x_t])$ | How much past to expose for candidate |
| **Candidate** | $\tilde{h}_t = \tanh(W \cdot [r_t \odot h_{t-1}, x_t])$ | New hidden state proposal |

**Hidden state update:** $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$

No separate cell state -- the hidden state does everything. Fewer parameters means faster training.

> **When to choose GRU vs LSTM:** GRU when you need speed and have less data. LSTM when you need maximum capacity for complex long-range dependencies.

In [ ]:
class GRUCell:
    def __init__(self, input_dim, hidden_dim, seed=42):
        rng = np.random.RandomState(seed)
        scale = 0.01
        combined = input_dim + hidden_dim
        self.W_z = rng.randn(combined, hidden_dim) * scale
        self.b_z = np.zeros(hidden_dim)
        self.W_r = rng.randn(combined, hidden_dim) * scale
        self.b_r = np.zeros(hidden_dim)
        self.W_h = rng.randn(combined, hidden_dim) * scale
        self.b_h = np.zeros(hidden_dim)
        self.hidden_dim = hidden_dim

    def step(self, x, h_prev):
        combined = np.concatenate([h_prev, x])
        z = sigmoid(combined @ self.W_z + self.b_z)
        r = sigmoid(combined @ self.W_r + self.b_r)
        combined_reset = np.concatenate([r * h_prev, x])
        h_candidate = np.tanh(combined_reset @ self.W_h + self.b_h)
        h_next = (1 - z) * h_prev + z * h_candidate
        return h_next

class GRUClassifier:
    def __init__(self, vocab_size, embed_dim, hidden_dim, seed=42):
        rng = np.random.RandomState(seed)
        self.embedding = rng.randn(vocab_size, embed_dim) * 0.01
        self.gru = GRUCell(embed_dim, hidden_dim, seed=seed)
        self.W_out = rng.randn(hidden_dim, 1) * 0.01
        self.b_out = np.zeros(1)
        self.hidden_dim = hidden_dim

    def forward(self, sequence):
        h = np.zeros(self.hidden_dim)
        for token in sequence:
            x = self.embedding[token]
            h = self.gru.step(x, h)
        logit = h @ self.W_out + self.b_out
        return sigmoid(logit[0]), h

gru_model = GRUClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
prob_gru, h_gru = gru_model.forward(X_train_seq[0])
print(f"GRU prediction probability: {prob_gru:.4f}")
print(f"Final hidden state norm:    {np.linalg.norm(h_gru):.4f}")

## STEP 13 -- Architecture Comparison

The chart below shows **representative** training curves for vanilla RNN, LSTM, and GRU on a sentiment classification task. These are simulated to illustrate the typical pattern:

- **Vanilla RNN** converges slowly and plateaus at lower accuracy (cannot capture long-range patterns)
- **LSTM** converges faster and reaches higher accuracy (cell state preserves information)
- **GRU** performs comparably to LSTM with slightly faster convergence (fewer parameters)

| Architecture | Parameters | Long-range | Training speed | Best for |
|-------------|-----------|------------|---------------|----------|
| Vanilla RNN | Fewest | Poor | Fastest per step | Short sequences, simple patterns |
| LSTM | Most | Excellent | Slowest per step | Complex dependencies, long sequences |
| GRU | Middle | Very good | Middle | Speed-accuracy trade-off |

> **Connecting to Friday:** The encoder-decoder you saw on Friday used LSTMs because it needed to compress an entire input sequence into a context vector -- long-range dependency is critical. Now you know what is inside those LSTM boxes: forget, input, cell state, and output gates working together to preserve information across many time steps.

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, 21))
losses_rnn  = [0.7 - 0.015*e + 0.002*e**0.5 for e in epochs]
losses_lstm = [0.7 - 0.025*e + 0.003*e**0.5 for e in epochs]
losses_gru  = [0.7 - 0.027*e + 0.003*e**0.5 for e in epochs]

accs_rnn  = [min(0.5 + 0.02*e - 0.0005*e**2, 0.85) for e in epochs]
accs_lstm = [min(0.5 + 0.025*e - 0.0005*e**2, 0.95) for e in epochs]
accs_gru  = [min(0.5 + 0.024*e - 0.0005*e**2, 0.94) for e in epochs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, losses_rnn, "o-", label="Vanilla RNN", markersize=3)
ax1.plot(epochs, losses_lstm, "s-", label="LSTM", markersize=3)
ax1.plot(epochs, losses_gru, "^-", label="GRU", markersize=3)
ax1.set_title("Training Loss (Representative)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, accs_rnn, "o-", label="Vanilla RNN", markersize=3)
ax2.plot(epochs, accs_lstm, "s-", label="LSTM", markersize=3)
ax2.plot(epochs, accs_gru, "^-", label="GRU", markersize=3)
ax2.set_title("Test Accuracy (Representative)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# STAGE 3 -- CNN Evaluation + MLOps Concepts

**Connecting to Friday:** We trained a CNN image classifier on CIFAR-10 using JumpStart. Now we deploy it, invoke it with test images, and evaluate per-class performance. Then we step back and ask: how do we stop doing all of this manually?

## STEP 14 -- Deploy and Evaluate the CNN

We locate the CNN model artifact from Friday and deploy it. JumpStart models deploy differently from Script Mode models -- the container and inference logic are provided by SageMaker.

While the endpoint deploys, we prepare CIFAR-10 test data for evaluation.

> **Note:** If the CNN model artifact is not available from Friday, the cell below provides a conceptual walkthrough instead of a live deployment. The evaluation concepts still apply.

In [ ]:
cnn_model_uri = None
for page in paginator.paginate(Bucket=bucket, Prefix=f"{prefix}/cnn-output/"):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith("model.tar.gz"):
            cnn_model_uri = f"s3://{bucket}/{obj['Key']}"

if not cnn_model_uri:
    for page in paginator.paginate(Bucket=bucket, Prefix=f"{prefix}/output/"):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith("model.tar.gz") and "cnn" in obj["Key"].lower():
                cnn_model_uri = f"s3://{bucket}/{obj['Key']}"

if cnn_model_uri:
    print(f"CNN model artifact found: {cnn_model_uri}")
    print("Proceeding with live deployment and evaluation.")
else:
    print("CNN model artifact NOT FOUND in S3.")
    print("This is expected if Friday's CNN training job was not completed.")
    print("We will do a conceptual walkthrough of CNN evaluation instead.")

### Conceptual CNN Evaluation

Whether or not we have a live CNN endpoint, the evaluation process is the same:

1. **Prepare test data:** Load CIFAR-10 test images (10,000 images, 10 classes)
2. **Send images to endpoint:** Each image is serialized as JPEG bytes and sent via `invoke_endpoint`
3. **Collect predictions:** The endpoint returns class probabilities; take the argmax
4. **Compute metrics:** Per-class accuracy, overall accuracy, confusion matrix

The confusion matrix for image classification reveals which classes the model confuses. For example, "cat" and "dog" share similar features and are commonly confused. This guides data collection and augmentation strategies.

| Metric | Image Classification Use |
|--------|------------------------|
| Per-class accuracy | Identify weak classes that need more training data |
| Confusion matrix | Find commonly confused class pairs |
| Top-1 accuracy | Overall model quality |
| Top-5 accuracy | Relevant for large class counts (ImageNet) |

In [ ]:
CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

np.random.seed(42)
n_test = 200
simulated_true = np.random.randint(0, 10, n_test)
simulated_pred = simulated_true.copy()
noise_idx = np.random.choice(n_test, size=n_test // 5, replace=False)
simulated_pred[noise_idx] = np.random.randint(0, 10, size=len(noise_idx))

cm_cnn = confusion_matrix(simulated_true, simulated_pred, labels=range(10))

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(cm_cnn, display_labels=CLASSES)
disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation=45)
ax.set_title("CIFAR-10 CNN -- Confusion Matrix (Simulated)")
plt.tight_layout()
plt.show()

per_class_acc = cm_cnn.diagonal() / cm_cnn.sum(axis=1)
print("Per-class accuracy:")
for cls, acc in zip(CLASSES, per_class_acc):
    print(f"  {cls:12s} {acc:.2%}")
print(f"\nOverall accuracy: {np.trace(cm_cnn) / cm_cnn.sum():.2%}")

## STEP 15 -- MLOps: From Manual to Automated

Today you performed the full ML workflow manually:

1. Located model artifacts in S3
2. Registered the model in Model Registry
3. Approved the model version
4. Deployed to an endpoint
5. Invoked the endpoint with test data
6. Evaluated performance
7. Cleaned up resources

This works for a one-time experiment. But FraudShield retrains weekly as new transaction data arrives. Doing this manually every week is error-prone and does not scale.

**MLOps** automates the ML lifecycle with three pillars:

| Pillar | What it does | Tools |
|--------|-------------|-------|
| **Automation** | Pipelines replace manual steps | SageMaker Pipelines, Step Functions |
| **Versioning & Governance** | Every model, dataset, and config is tracked | Model Registry, Experiments, Git |
| **Monitoring & Feedback** | Detect degradation, trigger retraining | Model Monitor, EventBridge |

### CI/CD vs ML CI/CD

| Traditional CI/CD | ML CI/CD |
|-------------------|----------|
| Code commit triggers build | Data change OR code change triggers retrain |
| Unit tests gate merge | Model evaluation gates registration |
| Binary artifact (JAR, Docker image) | Model artifact (`model.tar.gz`) |
| Application monitoring | Model quality + data quality monitoring |
| Bug fix | Drift detection triggers retraining |

## STEP 16 -- SageMaker Pipelines Concepts

A SageMaker Pipeline is a **directed acyclic graph (DAG)** of processing steps. Each step does one thing: preprocess data, train a model, evaluate metrics, register a version. Steps are connected by data dependencies -- the output of one step flows into the input of the next.

**FraudShield pipeline example:**

```
Preprocess --> Train --> Evaluate --> [F1 >= 0.85?] --YES--> Register
                                          |
                                         NO --> Stop (model not good enough)
```

**Key concepts:**

| Concept | What it is |
|---------|------------|
| `ProcessingStep` | Runs a preprocessing or evaluation script |
| `TrainingStep` | Runs a training job |
| `RegisterModel` | Registers a model version in Model Registry |
| `ConditionStep` | Branches the DAG based on a condition (e.g., F1 threshold) |
| `PropertyFile` | Captures output metrics from a step for use in conditions |
| `ParameterString` | Pipeline parameters that can be changed between runs |

SageMaker infers the DAG structure from how steps reference each other's outputs. You do not draw the graph manually.

> **Discussion:** Which step in the pipeline above prevents a bad model from reaching production? (The ConditionStep. If F1 is below the threshold, the model is never registered and never deployed.)

We will build an actual pipeline later in the curriculum. For now, understand the concepts and how they connect to what you did manually today.

## STEP 17 -- Mandatory Cleanup

Delete any remaining endpoints, endpoint configurations, and models. Check billing.

> **Make this a habit:** every time you deploy anything, verify cleanup before leaving.

In [ ]:
endpoints = sm_client.list_endpoints(StatusEquals="InService")["Endpoints"]
fraudshield_endpoints = [ep for ep in endpoints if "fraudshield" in ep["EndpointName"]]

if fraudshield_endpoints:
    for ep in fraudshield_endpoints:
        name = ep["EndpointName"]
        print(f"Deleting endpoint: {name}")
        sm_client.delete_endpoint(EndpointName=name)
    print("\nAll FraudShield endpoints deleted.")
else:
    print("No active FraudShield endpoints found. Cleanup complete.")

print("\nRemember to check Billing & Cost Management in the AWS Console.")

---
## Wrap-up

### What you accomplished today:

1. **Deployed and evaluated** the Random Forest fraud model from Friday -- you now know its precision, recall, F1, and where it makes mistakes (confusion matrix).
2. **Opened the LSTM black box** -- you implemented a vanilla RNN, saw why it fails (vanishing gradients), and built LSTM and GRU cells from scratch to understand how gates solve the problem.
3. **Explored CNN evaluation** -- per-class accuracy and confusion matrices for image classification.
4. **Learned MLOps concepts** -- how pipelines, registries, and monitoring automate the manual workflow you just performed.

### Tuesday Preview:

Tomorrow covers **data engineering**: Feature Store, Data Wrangler, Canvas, and Autopilot. The question shifts from "how do we deploy a model" to "how do we manage the features that feed it." Read the Advanced SageMaker Data concept threads before Tuesday.